# PathSnap on Google Colab

This notebook contains the Python implementation for each stage. It runs without a PathSnap checkout or preprocessed files. Open it in Colab, upload the original `Student folder.zip` using the Files sidebar (or point to a Google Drive copy), and run the cells in order. The archive or extracted folder must contain the `train` and `student_test` folders.

By default it prepares the data, trains logistic regression, and plots one student-test trip. To run another model, change `RUN_MODELS` in Setup and run its dedicated cell. Audit, HGB Bayesian optimisation, and full decoder evaluation are optional. KNN and forest can take substantially longer than the baseline.

In [ ]:
# Run once in Colab. The active kernel's Python receives the dependencies.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy>=1.26,<3', 'scikit-learn>=1.6,<2',
                'matplotlib>=3.8', 'optuna>=4,<6'], check=True)

In [ ]:
import os
import zipfile
from pathlib import Path

DATASET = Path('/content/Student folder.zip')
ROOT = Path('/content/pathsnap')
REBUILD_DATA = False
RUN_MODELS = {'logistic'}  # Choose any of: logistic, knn, forest, boosting, mlp, tree.
RUN_AUDIT = False
RUN_HGB_TUNING = False
RUN_DECODER_EVALUATION = False
RUN_ROUTE_PREVIEW = True
HGB_TRIALS = 30

ROOT.mkdir(parents=True, exist_ok=True)

def run_stage(entry, *args):
    previous_dir, previous_argv = Path.cwd(), sys.argv[:]
    try:
        os.chdir(ROOT)
        sys.argv = [entry.__name__, *map(str, args)]
        entry()
    finally:
        sys.argv = previous_argv
        os.chdir(previous_dir)

def dataset_root(path):
    if not path.exists():
        raise FileNotFoundError(f'Upload the original dataset or update DATASET: {path}')
    if path.is_file():
        destination = ROOT / 'raw'
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(path) as archive:
            for item in archive.infolist():
                if not (destination / item.filename).resolve().is_relative_to(destination.resolve()):
                    raise ValueError(f'Unsafe archive path: {item.filename}')
            archive.extractall(destination)
        path = destination
    for candidate in [path, *sorted(p for p in path.rglob('train') if p.is_dir())]:
        candidate = candidate.parent if candidate.name == 'train' else candidate
        if all((candidate / split).is_dir() for split in ('train', 'student_test')):
            return candidate.resolve()
    raise ValueError('Could not find train and student_test folders in the dataset.')

def ready(stage, filename):
    return all(sum(1 for _ in (ROOT / stage / split).glob(filename)) == count
               for split, count in [('train', 80), ('student_test', 10)])

## 1. Preprocess raw data

Read GPS, nodes, directed arcs and reference routes. Validate IDs and project geographic coordinates into local metres.

In [ ]:
"""Validate PathSnap trips and export label-separated NumPy artifacts."""
import argparse
import csv
import hashlib
import json
from pathlib import Path

import numpy as np


def read_table(path, columns):
    a = np.loadtxt(path, dtype=np.float64, ndmin=2)
    if a.size == 0 or a.shape[1] != columns:
        raise ValueError(f'{path.name}: expected nonempty {columns}-column table')
    return a


def valid_coordinates(a):
    return np.isfinite(a).all(axis=1) & (np.abs(a[:, 0]) <= 180) & (np.abs(a[:, 1]) <= 90)


def project(coords, origin):
    """Local equirectangular coordinates in metres, suitable for local trips."""
    delta = np.radians(coords - origin)
    return delta * np.array([6371008.8 * np.cos(np.radians(origin[1])), 6371008.8])


def components(n, arcs):
    parent = list(range(n))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i
    for u, v in arcs:
        parent[root(int(u))] = root(int(v))
    return len({root(i) for i in range(n)})


def process_trip(trip, destination):
    ident = trip.name
    paths = {ext: trip / f'{ident}.{ext}' for ext in ('track', 'nodes', 'arcs', 'route')}
    track = read_table(paths['track'], 3)
    nodes = read_table(paths['nodes'], 2)
    raw_arcs = read_table(paths['arcs'], 2)
    raw_route = read_table(paths['route'], 1).ravel()
    if not valid_coordinates(nodes).all():
        raise ValueError('Invalid network coordinates; refusing to renumber nodes')
    if not np.isfinite(raw_arcs).all() or not np.equal(raw_arcs, np.floor(raw_arcs)).all():
        raise ValueError('Arc endpoints must be finite integers')
    if (raw_arcs < 0).any() or (raw_arcs >= len(nodes)).any():
        raise ValueError('Arc endpoint outside node range')
    arcs = raw_arcs.astype(np.int64)
    if not np.isfinite(raw_route).all() or not np.equal(raw_route, np.floor(raw_route)).all():
        raise ValueError('Route arc IDs must be finite integers')
    if (raw_route < 0).any() or (raw_route >= len(arcs)).any():
        raise ValueError('Reference arc ID outside arc range')
    route = raw_route.astype(np.int64)
    mask = valid_coordinates(track[:, :2]) & np.isfinite(track[:, 2])
    clean = track[mask]
    if not len(clean):
        raise ValueError('No valid GPS observations')
    if (np.diff(clean[:, 2]) < 0).any():
        raise ValueError('Timestamps decrease; refusing to reorder the trajectory silently')
    origin = np.median(nodes, axis=0)
    gps_xy, nodes_xy = project(clean[:, :2], origin), project(nodes, origin)
    dt = np.diff(clean[:, 2])
    distance = np.linalg.norm(np.diff(gps_xy, axis=0), axis=1)
    speed = np.divide(distance, dt, out=np.full(len(dt), np.nan), where=dt > 0)
    lengths = np.linalg.norm(nodes_xy[arcs[:, 1]] - nodes_xy[arcs[:, 0]], axis=1)
    row = dict(trip_id=ident, raw_observations=len(track), observations=len(clean),
               removed_observations=int((~mask).sum()), nodes=len(nodes), arcs=len(arcs),
               route_arcs=len(route), duplicate_timestamps=int((dt == 0).sum()),
               gaps_over_1s=int((dt > 1.000001).sum()), max_gap_s=float(dt.max()) if len(dt) else 0,
               zero_length_arcs=int((lengths == 0).sum()), weak_components=components(len(nodes), arcs),
               isolated_nodes=int(len(nodes) - len(np.unique(arcs))),
               reference_disconnections=int((arcs[route[:-1], 1] != arcs[route[1:], 0]).sum()),
               max_speed_mps=float(np.nanmax(speed)) if np.isfinite(speed).any() else None,
               extent_m=float(np.linalg.norm(np.ptp(nodes_xy, axis=0))))
    destination.mkdir(parents=True, exist_ok=True)
    # Inference assets deliberately contain no route/reference values.
    np.savez_compressed(destination / 'inputs.npz', gps_lonlat=clean[:, :2], timestamps=clean[:, 2],
                        observation_ids=np.flatnonzero(mask), gps_xy=gps_xy, nodes_lonlat=nodes,
                        nodes_xy=nodes_xy, arcs=arcs, arc_ids=np.arange(len(arcs)),
                        arc_length_m=lengths, projection_origin_lonlat=origin)
    labels = destination.parent / '_references'
    labels.mkdir(exist_ok=True)
    np.savez_compressed(labels / f'{ident}.npz', route_arc_ids=route)
    metadata = dict(summary=row, source_sha256={k: hashlib.sha256(p.read_bytes()).hexdigest() for k,p in paths.items()},
                    projection='local equirectangular; R=6371008.8m; origin=median network lon/lat',
                    policy='Preserve node/arc row IDs and observation order. Remove invalid GPS rows only; retain duplicate timestamps and gaps. Route is a sequence, not pointwise labels.')
    (destination / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    return row


def preprocess_main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--data-root', type=Path, required=True)
    parser.add_argument('--output', type=Path, default=Path('data/processed'))
    parser.add_argument('--report-dir', type=Path, default=Path('results/preprocessing'))
    args = parser.parse_args()
    rows, failures, seen = [], [], set()
    for partition, expected in [('train', 80), ('student_test', 10)]:
        trips = sorted(p for p in (args.data_root / partition).iterdir() if p.is_dir())
        if len(trips) != expected:
            raise ValueError(f'{partition}: expected {expected} trips, found {len(trips)}')
        for trip in trips:
            if trip.name in seen:
                raise ValueError(f'Trip overlaps partitions: {trip.name}')
            seen.add(trip.name)
            try:
                row = process_trip(trip, args.output / partition / trip.name)
                rows.append(dict(partition=partition, **row))
            except (ValueError, OSError) as exc:
                failures.append(dict(partition=partition, trip_id=trip.name, error=str(exc)))
    args.report_dir.mkdir(parents=True, exist_ok=True)
    if rows:
        with (args.report_dir / 'inventory.csv').open('w', newline='', encoding='utf-8') as handle:
            writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    summary = dict(successful_trips=len(rows), failed_trips=failures,
                   observations=sum(r['observations'] for r in rows),
                   removed_observations=sum(r['removed_observations'] for r in rows),
                   duplicate_timestamps=sum(r['duplicate_timestamps'] for r in rows),
                   gaps_over_1s=sum(r['gaps_over_1s'] for r in rows))
    (args.report_dir / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2))
    if failures:
        raise SystemExit(1)

In [ ]:
if REBUILD_DATA or not ready('data/processed', '*/inputs.npz'):
    run_stage(preprocess_main, '--data-root', dataset_root(DATASET))
else:
    print('Using existing processed trips. Set REBUILD_DATA = True for a different dataset.')

## 2. Extract candidate roads and features

Generate nearby road candidates and compute the 12 model inputs. Candidate metadata preserves the original road and observation IDs.

In [ ]:
"""Generate directed road candidates and reference-free geometric features."""
import argparse
import csv
import hashlib
import json
import time
from collections import defaultdict
from pathlib import Path

import numpy as np

FEATURE_NAMES = ['distance_m', 'distance_rank', 'projection_fraction', 'arc_length_m',
                 'heading_cosine', 'heading_available', 'previous_step_m',
                 'previous_dt_s', 'previous_speed_mps', 'gap_over_5s',
                 'start_out_degree', 'end_in_degree']


def project_to_segments(point, starts, vectors, squared_lengths):
    fraction = np.divide(np.sum((point - starts) * vectors, axis=1), squared_lengths,
                         out=np.zeros(len(starts)), where=squared_lengths > 0)
    fraction = np.clip(fraction, 0, 1)
    snapped = starts + fraction[:, None] * vectors
    return np.linalg.norm(snapped - point, axis=1), fraction, snapped


def nearest_order(distances, arc_ids, k):
    """Order the closest k arcs, resolving cutoff ties by arc ID."""
    if len(distances) <= k:
        return np.lexsort((arc_ids, distances))
    cutoff = np.partition(distances, k - 1)[k - 1]
    # Distances equal to the cutoff must be considered before breaking ties.
    eligible = np.flatnonzero(distances <= cutoff)
    return eligible[np.lexsort((arc_ids[eligible], distances[eligible]))[:k]]


def generate(inputs, radius=100.0, k=8):
    if not np.isfinite(radius) or radius <= 0 or k < 1:
        raise ValueError('radius must be positive and k must be at least one')
    points, timestamps = inputs['gps_xy'], inputs['timestamps']
    arcs, nodes = inputs['arcs'], inputs['nodes_xy']
    arc_ids = inputs['arc_ids']
    if not len(arcs):
        raise ValueError('No network arcs')
    starts = nodes[arcs[:, 0]]
    vectors = nodes[arcs[:, 1]] - starts
    squared = np.sum(vectors * vectors, axis=1)
    lengths = np.sqrt(squared)
    # Insert each radius-expanded segment bounding box into a uniform grid.
    # Every segment within radius of a point is guaranteed to be in its cell.
    grid = defaultdict(list)
    low = np.floor((np.minimum(starts, starts + vectors) - radius) / radius).astype(int)
    high = np.floor((np.maximum(starts, starts + vectors) + radius) / radius).astype(int)
    for arc, (lo, hi) in enumerate(zip(low, high)):
        for x in range(lo[0], hi[0] + 1):
            for y in range(lo[1], hi[1] + 1):
                grid[x, y].append(arc)
    grid = {cell: np.array(ids, dtype=np.int64) for cell, ids in grid.items()}
    outgoing = np.bincount(arcs[:, 0], minlength=len(nodes))
    incoming = np.bincount(arcs[:, 1], minlength=len(nodes))
    features, selected, snapped_points, offsets, fallbacks = [], [], [], [0], []
    all_arcs = np.arange(len(arcs))
    empty = np.empty(0, dtype=np.int64)
    for i, point in enumerate(points):
        cell = tuple(np.floor(point / radius).astype(int))
        pool = grid.get(cell, empty)
        distances, fractions, snaps = project_to_segments(point, starts[pool], vectors[pool], squared[pool])
        keep = distances <= radius
        fallback = not keep.any()
        if fallback:
            pool = all_arcs
            distances, fractions, snaps = project_to_segments(point, starts, vectors, squared)
        else:
            pool, distances, fractions, snaps = pool[keep], distances[keep], fractions[keep], snaps[keep]
        # Ties are deterministic and preserve directed arc identity.
        order = nearest_order(distances, arc_ids[pool], k)
        ids, ds, fs, ss = pool[order], distances[order], fractions[order], snaps[order]
        dt = float(timestamps[i] - timestamps[i - 1]) if i else 0.0
        movement = point - points[i - 1] if i else np.zeros(2)
        step = float(np.linalg.norm(movement))
        heading_ok = i > 0 and 0 < dt <= 5 and step >= 2
        heading_available = heading_ok & (lengths[ids] > 0)
        cosine = np.zeros(len(ids))
        if heading_ok:
            cosine = np.divide(vectors[ids] @ movement, lengths[ids] * step,
                               out=cosine, where=heading_available)
        matrix = np.column_stack([ds, np.arange(len(ids)), fs, lengths[ids], cosine,
                                  heading_available, np.full(len(ids), step),
                                  np.full(len(ids), dt), np.full(len(ids), step / dt if dt > 0 else 0),
                                  np.full(len(ids), dt > 5), outgoing[arcs[ids, 0]], incoming[arcs[ids, 1]]])
        features.append(matrix)
        selected.extend(arc_ids[ids])
        snapped_points.extend(ss)
        offsets.append(offsets[-1] + len(ids))
        fallbacks.append(fallback)
    result = dict(features=np.concatenate(features).astype(np.float32),
                  candidate_arc_ids=np.array(selected, dtype=np.int64),
                  snapped_xy=np.array(snapped_points), offsets=np.array(offsets, dtype=np.int64),
                  observation_ids=inputs['observation_ids'], fallback_observations=np.array(fallbacks),
                  feature_names=np.array(FEATURE_NAMES))
    if not np.isfinite(result['features']).all():
        raise ValueError('Nonfinite feature value')
    return result


def candidates_main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--input-root', type=Path, default=Path('data/processed'))
    parser.add_argument('--output', type=Path, default=Path('data/features'))
    parser.add_argument('--report-dir', type=Path, default=Path('results/features'))
    parser.add_argument('--radius', type=float, default=100)
    parser.add_argument('--k', type=int, default=8)
    args = parser.parse_args()
    rows = []
    for partition in ['train', 'student_test']:
        paths = sorted((args.input_root / partition).glob('*/inputs.npz'))
        if not paths:
            raise ValueError(f'No input trips in {partition}')
        for path in paths:
            started = time.perf_counter()
            with np.load(path, allow_pickle=False) as inputs:
                result = generate(inputs, args.radius, args.k)
            target = args.output / partition / path.parent.name
            target.mkdir(parents=True, exist_ok=True)
            np.savez_compressed(target / 'candidates.npz', **result)
            rows.append(dict(partition=partition, trip_id=path.parent.name,
                             observations=len(result['observation_ids']), candidates=len(result['features']),
                             fallback_observations=int(result['fallback_observations'].sum()),
                             seconds=round(time.perf_counter() - started, 3)))
            (target / 'metadata.json').write_text(json.dumps(dict(radius_m=args.radius, k=args.k,
                source_sha256=hashlib.sha256(path.read_bytes()).hexdigest(), feature_names=FEATURE_NAMES), indent=2))
            print(f"{partition}/{path.parent.name}: {rows[-1]['candidates']} candidates", flush=True)
    args.report_dir.mkdir(parents=True, exist_ok=True)
    with (args.report_dir / 'inventory.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    summary = dict(trips=len(rows), observations=sum(r['observations'] for r in rows),
                   candidates=sum(r['candidates'] for r in rows),
                   fallback_observations=sum(r['fallback_observations'] for r in rows),
                   radius_m=args.radius, max_candidates=args.k, feature_names=FEATURE_NAMES)
    (args.report_dir / 'summary.json').write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))

In [ ]:
if REBUILD_DATA or not ready('data/features', '*/candidates.npz'):
    run_stage(candidates_main)

## 3. Align the reference routes and optionally audit the candidates

The ordered reference route provides approximate point labels. The 100 m alignment filter selects which observations can be scored.

In [ ]:
"""Align GPS observations to an ordered reference route for supervision."""
import argparse
import json
from pathlib import Path

import numpy as np


def align(points, starts, ends):
    """Minimize total clipped squared distance with nondecreasing route indices."""
    vectors = ends - starts
    squared = np.sum(vectors ** 2, axis=1)
    positions = np.arange(len(starts), dtype=np.int32)
    back = np.empty((len(points), len(starts)), dtype=np.int32)
    previous = np.zeros(len(starts))
    for i, point in enumerate(points):
        prefix = np.minimum.accumulate(previous)
        best = np.maximum.accumulate(np.where(
            np.r_[True, prefix[1:] < prefix[:-1]], positions, 0))
        back[i] = best
        offset = point - starts
        fraction = np.clip(np.divide(np.sum(offset * vectors, axis=1), squared,
                                     out=np.zeros_like(squared), where=squared > 0), 0, 1)
        residual = offset - fraction[:, None] * vectors
        previous = prefix + np.minimum(np.sum(residual ** 2, axis=1), 150 ** 2)
    indices = np.empty(len(points), dtype=np.int32)
    indices[-1] = previous.argmin()
    for i in range(len(points) - 1, 0, -1):
        indices[i - 1] = back[i, indices[i]]
    diff = points - starts[indices]
    chosen_vectors, chosen_sq = vectors[indices], squared[indices]
    fraction = np.clip(np.divide(np.sum(diff * chosen_vectors, axis=1), chosen_sq,
                                 out=np.zeros(len(points)), where=chosen_sq > 0), 0, 1)
    return indices, np.linalg.norm(diff - fraction[:, None] * chosen_vectors, axis=1)


def align_main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--inputs', type=Path, default=Path('data/processed'))
    parser.add_argument('--output', type=Path, default=Path('data/labels'))
    args = parser.parse_args()
    for split, expected in [('train', 80), ('student_test', 10)]:
        paths = sorted((args.inputs / split).glob('*/inputs.npz'))
        if len(paths) != expected:
            parser.error(f'{split}: expected {expected} trips, found {len(paths)}')
        dest = args.output / split
        dest.mkdir(parents=True, exist_ok=True)
        totals = dict(trips=len(paths), observations=0, within_100m=0)
        for path in paths:
            trip = path.parent.name
            with np.load(path) as inputs, np.load(path.parents[1] / '_references' / f'{trip}.npz') as ref:
                route = ref['route_arc_ids']
                endpoints = inputs['nodes_xy'][inputs['arcs'][route]]
                sequence, distance = align(inputs['gps_xy'], endpoints[:, 0], endpoints[:, 1])
                valid = distance <= 100
                np.savez_compressed(dest / f'{trip}.npz', observation_ids=inputs['observation_ids'],
                                    arc_ids=route[sequence], route_index=sequence,
                                    alignment_distance_m=distance, usable=valid)
            totals['observations'] += len(sequence)
            totals['within_100m'] += int(valid.sum())
        print(split, json.dumps(totals), flush=True)

In [ ]:
if REBUILD_DATA or not ready('data/labels', '*.npz'):
    run_stage(align_main)

In [ ]:
"""Check projection, network diagnostics, saved features and candidate recall."""
import argparse
import csv
import json
from pathlib import Path
from time import perf_counter

import numpy as np



def spherical_distance(start, end):
    a, b = np.radians(start), np.radians(end)
    lon, lat = (b - a).T
    h = np.sin(lat / 2) ** 2 + np.cos(a[:, 1]) * np.cos(b[:, 1]) * np.sin(lon / 2) ** 2
    return 2 * 6371008.8 * np.arcsin(np.sqrt(np.clip(h, 0, 1)))


def diagnostics(inputs):
    arcs = inputs['arcs']
    n = len(inputs['nodes_xy'])
    keys = arcs[:, 0] * n + arcs[:, 1]
    reverse = arcs[:, 1] * n + arcs[:, 0]
    incoming = np.bincount(arcs[:, 1], minlength=n)
    outgoing = np.bincount(arcs[:, 0], minlength=n)
    coords = inputs['nodes_lonlat']
    gps = inputs['gps_lonlat']
    times = np.diff(inputs['timestamps'])
    gps_dist = np.linalg.norm(np.diff(inputs['gps_xy'], axis=0), axis=1)
    road_error = np.abs(inputs['arc_length_m'] - spherical_distance(coords[arcs[:, 0]], coords[arcs[:, 1]]))
    gps_error = np.abs(gps_dist - spherical_distance(gps[:-1], gps[1:]))
    return dict(sources=int(((incoming == 0) & (outgoing > 0)).sum()),
                sinks=int(((outgoing == 0) & (incoming > 0)).sum()),
                reciprocal_arcs=int(np.count_nonzero((keys != reverse) & np.isin(reverse, keys))),
                duplicate_directed_arcs=int(len(keys) - len(np.unique(keys))),
                max_in_degree=int(incoming.max()), max_out_degree=int(outgoing.max()),
                road_length_max_error_m=float(road_error.max(initial=0)),
                road_length_p95_error_m=float(np.quantile(road_error, 0.95)),
                gps_step_max_error_m=float(gps_error.max(initial=0)),
                gps_step_p95_error_m=float(np.quantile(gps_error, 0.95)) if len(gps_error) else 0,
                nonpositive_dt=int((times <= 0).sum()), gaps_over_5s=int((times > 5).sum()),
                max_gap_s=float(times.max(initial=0)))


def audit_search(inputs, saved, radius, k, rng, full_max_observations=0):
    points, arcs, nodes, ids = (inputs[key] for key in ('gps_xy', 'arcs', 'nodes_xy', 'arc_ids'))
    starts = nodes[arcs[:, 0]]
    vectors = nodes[arcs[:, 1]] - starts
    squared = np.sum(vectors ** 2, axis=1)
    fallback = saved['fallback_observations']
    exhaustive = len(points) <= full_max_observations
    sample = (np.arange(len(points)) if exhaustive else
              np.union1d(rng.choice(len(points), min(24, len(points)), replace=False), np.flatnonzero(fallback)))
    projection_s = full_sort_s = partial_sort_s = 0.0
    for i in sample:
        tick = perf_counter()
        distance, _, _ = project_to_segments(points[i], starts, vectors, squared)
        projection_elapsed = perf_counter() - tick
        pool = np.flatnonzero(distance <= radius)
        if bool(fallback[i]) != (len(pool) == 0):
            raise ValueError(f'Wrong fallback flag at observation {i}')
        if fallback[i]:
            projection_s += projection_elapsed
            tick = perf_counter()
            expected = np.lexsort((ids, distance))[:k]
            full_sort_s += perf_counter() - tick
            tick = perf_counter()
            actual = nearest_order(distance, ids, k)
            partial_sort_s += perf_counter() - tick
            np.testing.assert_array_equal(actual, expected)
        else:
            expected = pool[np.lexsort((ids[pool], distance[pool]))[:k]]
        left, right = saved['offsets'][i:i + 2]
        np.testing.assert_array_equal(saved['candidate_arc_ids'][left:right], ids[expected])
    return dict(search_checks=len(sample), exhaustive_trip=int(exhaustive), fallback_count=int(fallback.sum()),
                fallback_projection_seconds=projection_s, fallback_full_sort_seconds=full_sort_s,
                fallback_partial_sort_seconds=partial_sort_s)


def audit_main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--inputs', type=Path, default=Path('data/processed'))
    parser.add_argument('--features', type=Path, default=Path('data/features'))
    parser.add_argument('--labels', type=Path, default=Path('data/labels'))
    parser.add_argument('--output', type=Path, default=Path('results/feature_audit'))
    parser.add_argument('--full-max-observations', type=int, default=0,
                        help='Exhaustively check trips up to this size (0 samples every trip)')
    args = parser.parse_args()
    if args.full_max_observations < 0:
        parser.error('--full-max-observations must be nonnegative')
    rows = []
    rng = np.random.default_rng(557)
    for split, expected in [('train', 80), ('student_test', 10)]:
        paths = sorted((args.inputs / split).glob('*/inputs.npz'))
        if len(paths) != expected:
            parser.error(f'{split}: expected {expected} trips, found {len(paths)}')
        for path in paths:
            trip = path.parent.name
            folder = args.features / split / trip
            meta = json.loads((folder / 'metadata.json').read_text())
            with np.load(path) as file:
                inputs = dict(file)
            with np.load(folder / 'candidates.npz') as file:
                saved = dict(file)
            with np.load(args.labels / split / f'{trip}.npz') as file:
                labels = dict(file)
            radius, k = meta['radius_m'], meta['k']
            tick = perf_counter()
            regenerated = generate(inputs, radius, k)
            generation_s = perf_counter() - tick
            for key, value in regenerated.items():
                np.testing.assert_array_equal(saved[key], value, err_msg=f'{split}/{trip}: {key}')
            np.testing.assert_array_equal(saved['observation_ids'], labels['observation_ids'])
            offsets = saved['offsets']
            matches = saved['candidate_arc_ids'] == np.repeat(labels['arc_ids'], np.diff(offsets))
            covered = np.logical_or.reduceat(matches, offsets[:-1])
            usable = labels['usable']
            rows.append(dict(split=split, trip=trip, observations=len(usable),
                             aligned=int(usable.sum()), covered=int((covered & usable).sum()),
                             candidate_rows=len(saved['features']), generation_seconds=generation_s,
                             **diagnostics(inputs),
                             **audit_search(inputs, saved, radius, k, rng, args.full_max_observations)))
        print(f'{split}: audited {len(paths)} trips', flush=True)
    args.output.mkdir(parents=True, exist_ok=True)
    with (args.output / 'trips.csv').open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    summary = {}
    summed = ['observations', 'aligned', 'covered', 'candidate_rows', 'search_checks', 'fallback_count',
              'exhaustive_trip', 'generation_seconds', 'fallback_projection_seconds', 'fallback_full_sort_seconds',
              'fallback_partial_sort_seconds', 'nonpositive_dt', 'gaps_over_5s']
    for split in ('train', 'student_test'):
        group = [r for r in rows if r['split'] == split]
        stats = {key: sum(r[key] for r in group) for key in summed}
        stats.update({key: max(r[key] for r in group) for key in
                      ('max_gap_s', 'road_length_max_error_m', 'gps_step_max_error_m',
                       'road_length_p95_error_m', 'gps_step_p95_error_m')})
        stats['candidate_recall_aligned'] = stats['covered'] / stats['aligned']
        summary[split] = stats
    (args.output / 'summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(json.dumps(summary, indent=2))

In [ ]:
if RUN_AUDIT:
    run_stage(audit_main, '--full-max-observations', '500')

## 4. Define training and run each model

Each model cell uses the same trip-grouped folds and scores the student test split after the final fit. Enable one or more names in `RUN_MODELS` and run the corresponding cells.

In [ ]:
"""Run trip-grouped candidate classification experiments."""
import argparse
import json
import pickle
import time
from pathlib import Path

import numpy as np

MODELS = ['logistic', 'knn', 'forest', 'boosting', 'mlp', 'tree']
SEED = 557


def load_trips(root, labels, split, names):
    rows = []
    paths = sorted((root / split).glob('*/candidates.npz'))
    expected = {'train': 80, 'student_test': 10}[split]
    if len(paths) != expected:
        raise ValueError(f'{split}: expected {expected} feature files, found {len(paths)}')
    for path in paths:
        trip = path.parent.name
        with np.load(path) as f, np.load(labels / split / f'{trip}.npz') as y:
            if not np.array_equal(f['observation_ids'], y['observation_ids']):
                raise ValueError(f'Observation IDs disagree for {trip}')
            if not np.array_equal(f['feature_names'], names):
                raise ValueError(f'Feature schema disagrees for {trip}')
            counts = np.diff(f['offsets'])
            usable = y['usable']
            keep = np.repeat(usable, counts)
            target = np.repeat(y['arc_ids'], counts)
            matched = f['candidate_arc_ids'][keep] == target[keep]
            rows.append((trip, f['features'][keep], matched, counts[usable], len(counts)))
    return rows


def combine(trips):
    _, x, y, counts, totals = zip(*trips)
    return np.concatenate(x), np.concatenate(y), np.concatenate(counts), sum(totals)


def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -40, 40)))


def fit_logistic(x, y):
    # Standardize within the fit partition and use damped Newton steps.
    mean = x.mean(axis=0, dtype=np.float64)
    scale = x.std(axis=0, dtype=np.float64)
    scale[scale == 0] = 1
    design = np.empty((len(x), x.shape[1] + 1), dtype=np.float64)
    design[:, 0] = 1
    design[:, 1:] = np.clip((x - mean) / scale, -20, 20)
    weights = np.zeros(design.shape[1])
    regularization = np.full(len(weights), 0.001)
    regularization[0] = 0
    def objective(params):
        logits = design @ params
        return np.mean(np.logaddexp(0, logits) - y * logits) + 0.5 * np.dot(regularization, params ** 2)
    for _ in range(20):
        p = sigmoid(design @ weights)
        gradient = design.T @ (p - y) / len(x) + regularization * weights
        if np.max(np.abs(gradient)) < 1e-6:
            break
        curvature = p * (1 - p)
        hessian = (design.T * curvature) @ design / len(x) + np.diag(regularization)
        step = np.linalg.solve(hessian, gradient)
        loss = objective(weights)
        for damping in 0.5 ** np.arange(20):
            trial = weights - damping * step
            if objective(trial) < loss:
                weights = trial
                break
        else:
            break
    return dict(mean=mean, scale=scale, weights=weights)


def fit(name, x, y, jobs):
    if not y.any() or y.all():
        raise ValueError('Training fold needs both candidate classes')
    if name == 'logistic':
        return fit_logistic(x, y), dict(l2=0.001, max_iter=20, standardized_clip=20)
    from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
    from sklearn.neighbors import KNeighborsClassifier
    from sklearn.neural_network import MLPClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.tree import DecisionTreeClassifier

    if name == 'knn':
        classifier = KNeighborsClassifier(n_neighbors=15, weights='distance',
                                           algorithm='kd_tree', n_jobs=jobs)
        model = make_pipeline(StandardScaler(), classifier)
        return model.fit(x, y), dict(scaling='StandardScaler', **classifier.get_params())
    if name == 'tree':
        model = DecisionTreeClassifier(max_depth=8, min_samples_leaf=40, random_state=SEED)
    elif name == 'forest':
        model = RandomForestClassifier(n_estimators=40, max_depth=14, max_samples=0.25,
                                       min_samples_leaf=20, n_jobs=jobs, random_state=SEED)
    elif name == 'boosting':
        model = HistGradientBoostingClassifier(max_iter=120, max_leaf_nodes=31,
                                               early_stopping=False, random_state=SEED)
    else:
        from sklearn.exceptions import ConvergenceWarning
        import warnings
        classifier = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=15, batch_size=2048,
                                   early_stopping=False, random_state=SEED)
        model = make_pipeline(StandardScaler(), classifier)
        with warnings.catch_warnings():
            # A fixed epoch budget is intentional. Keep the actual loss history.
            warnings.filterwarnings('ignore', category=ConvergenceWarning)
            model.fit(x, y)
        return model, dict(scaling='StandardScaler', **classifier.get_params())
    return model.fit(x, y), model.get_params()


def scores(name, model, x):
    if name == 'logistic':
        z = np.clip((x - model['mean']) / model['scale'], -20, 20)
        return sigmoid(z @ model['weights'][1:] + model['weights'][0])
    return model.predict_proba(x)[:, 1]


def metrics(prob, target, counts, total):
    starts = np.r_[0, np.cumsum(counts[:-1])]
    highest = np.maximum.reduceat(prob, starts)
    indices = np.arange(len(prob))
    # On ties choose the closest candidate using the existing candidate order.
    choices = np.minimum.reduceat(np.where(prob == np.repeat(highest, counts), indices, len(prob)), starts)
    hits = int(target[choices].sum())
    covered = int(np.count_nonzero(np.logical_or.reduceat(target, starts)))
    tp = hits
    fp = len(counts) - hits
    fn = int(target.sum()) - tp
    tn = len(target) - tp - fp - fn
    p = np.clip(prob.astype(float), 1e-8, 1 - 1e-8)
    return dict(observations=total, aligned_observations=len(counts), candidate_covered=covered,
                candidate_miss_rate=1 - covered / len(counts), top1_accuracy=hits / len(counts),
                top1_when_covered=hits / covered if covered else None,
                macro_f1=0.5 * (2 * tp / max(1, 2 * tp + fp + fn) + 2 * tn / max(1, 2 * tn + fp + fn)),
                binary_log_loss=float(-np.mean(np.where(target, np.log(p), np.log1p(-p)))),
                confusion=dict(tp=tp, fp=fp, fn=fn, tn=tn))


def evaluate(name, model, dataset):
    x, y, counts, total = dataset
    start = time.perf_counter()
    prob = scores(name, model, x)
    return dict(inference_seconds=time.perf_counter() - start, **metrics(prob, y, counts, total))


def training_main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--features', type=Path, default=Path('data/features'))
    parser.add_argument('--labels', type=Path, default=Path('data/labels'))
    parser.add_argument('--output', type=Path, default=Path('results/training'),
                        help='Directory for evaluation reports')
    parser.add_argument('--model-dir', type=Path, default=Path('models'),
                        help='Directory for trained model artifacts')
    parser.add_argument('--models', nargs='+', choices=MODELS, default=MODELS)
    parser.add_argument('--folds', type=int, default=3, choices=range(2, 6))
    parser.add_argument('--jobs', type=int, default=2)
    args = parser.parse_args()
    if set(args.models) - {'logistic'}:
        try:
            import sklearn
        except ImportError:
            parser.error('Install requirements-training.txt or use --models logistic.')
    first = next((args.features / 'train').glob('*/candidates.npz'), None)
    if first is None:
        parser.error('Run candidate generation before training.')
    with np.load(first) as f:
        names = f['feature_names'].tolist()
    train = load_trips(args.features, args.labels, 'train', names)
    student = load_trips(args.features, args.labels, 'student_test', names)
    if {t[0] for t in train} & {t[0] for t in student}:
        raise ValueError('Train and student_test trip IDs overlap')
    folds = [[] for _ in range(args.folds)]
    sizes = [0] * args.folds
    shuffled = np.random.default_rng(SEED).permutation(len(train))
    for i in sorted(shuffled, key=lambda i: len(train[i][3]), reverse=True):
        fold = int(np.argmin(sizes))
        folds[fold].append(train[i])
        sizes[fold] += len(train[i][3])
    args.output.mkdir(parents=True, exist_ok=True)
    args.model_dir.mkdir(parents=True, exist_ok=True)
    test = combine(student)
    _, y, counts, total = test
    nearest = np.zeros(len(y))
    nearest[np.r_[0, np.cumsum(counts[:-1])]] = 1
    baseline = metrics(nearest, y, counts, total)
    del baseline['binary_log_loss']  # Baseline scores are hard decisions.
    (args.output / 'nearest.json').write_text(json.dumps(baseline, indent=2))
    for name in dict.fromkeys(args.models):
        cv = []
        for i, validation in enumerate(folds):
            x, y, _, _ = combine([t for j, fold in enumerate(folds) if j != i for t in fold])
            started = time.perf_counter()
            model, _ = fit(name, x, y, args.jobs)
            elapsed = time.perf_counter() - started
            cv.append(dict(fold=i + 1, trip_ids=[t[0] for t in validation], fit_seconds=elapsed,
                           **evaluate(name, model, combine(validation))))
            print(name, 'fold', i + 1, 'top1', round(cv[-1]['top1_accuracy'], 4), flush=True)
            del x, y, model
        x, y, _, _ = combine(train)
        started = time.perf_counter()
        model, config = fit(name, x, y, args.jobs)
        fit_seconds = time.perf_counter() - started
        result = dict(model=name, configuration=config, seed=SEED, feature_names=names,
                      numpy_version=np.__version__, sklearn_version=sklearn.__version__ if name != 'logistic' else None,
                      folds=cv, cv_macro_f1=float(np.mean([f['macro_f1'] for f in cv])),
                      cv_top1=float(np.mean([f['top1_accuracy'] for f in cv])), final_fit_seconds=fit_seconds,
                      student_test=evaluate(name, model, test))
        if name == 'logistic':
            np.savez_compressed(args.model_dir / 'logistic.npz', **model, feature_names=names)
        else:
            with (args.model_dir / f'{name}.pkl').open('wb') as handle:
                pickle.dump(dict(model=model, feature_names=names, sklearn_version=sklearn.__version__), handle)
            if name == 'mlp':
                result['train_loss_by_epoch'] = model[-1].loss_curve_
        (args.output / f'{name}.json').write_text(json.dumps(result, indent=2))
        print(name, 'student_test', json.dumps(result['student_test']), flush=True)
        del x, y, model

### Logistic regression

In [ ]:
if 'logistic' in RUN_MODELS:
    run_stage(training_main, '--models', 'logistic')

### K-nearest neighbors

In [ ]:
if 'knn' in RUN_MODELS:
    run_stage(training_main, '--models', 'knn')

### Random Forest

In [ ]:
if 'forest' in RUN_MODELS:
    run_stage(training_main, '--models', 'forest')

### Histogram Gradient Boosting

In [ ]:
if 'boosting' in RUN_MODELS:
    run_stage(training_main, '--models', 'boosting')

### Small MLP

In [ ]:
if 'mlp' in RUN_MODELS:
    run_stage(training_main, '--models', 'mlp')

### Decision Tree

In [ ]:
if 'tree' in RUN_MODELS:
    run_stage(training_main, '--models', 'tree')

## 5. Optional Bayesian tuning for Histogram Gradient Boosting

The objective defaults to macro F1 across the training folds. The student-test split is scored after Optuna selects the settings.

In [ ]:
"""Tune PathSnap's Histogram Gradient Boosting using Optuna and trip-grouped cross-validation."""
import argparse
import json
import pickle
from pathlib import Path

import numpy as np
import sklearn
from sklearn.ensemble import HistGradientBoostingClassifier
from threadpoolctl import threadpool_limits


BASELINE = dict(max_iter=120, max_leaf_nodes=31, learning_rate=0.1,
                min_samples_leaf=20, l2_regularization=0.0)


def parameters(trial):
    return dict(max_iter=trial.suggest_int('max_iter', 60, 240),
                max_leaf_nodes=trial.suggest_int('max_leaf_nodes', 15, 63),
                learning_rate=trial.suggest_float('learning_rate', 0.02, 0.25, log=True),
                min_samples_leaf=trial.suggest_int('min_samples_leaf', 10, 160, log=True),
                l2_regularization=trial.suggest_float('l2_regularization', 0.0, 5.0))


def estimator(params):
    return HistGradientBoostingClassifier(random_state=SEED, early_stopping=False, **params)


def grouped_folds(trips):
    """Assign complete trips using the training runner's split procedure."""
    folds = [[], [], []]
    sizes = [0, 0, 0]
    shuffled = np.random.default_rng(SEED).permutation(len(trips))
    for i in sorted(shuffled, key=lambda i: len(trips[i][3]), reverse=True):
        fold = int(np.argmin(sizes))
        folds[fold].append(trips[i])
        sizes[fold] += len(trips[i][3])
    return folds


def hgb_tuning_main():
    import optuna
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--objective', choices=['macro_f1', 'top1_accuracy'], default='macro_f1')
    parser.add_argument('--trials', type=int, default=30)
    parser.add_argument('--threads', type=int, default=2)
    parser.add_argument('--features', type=Path, default=Path('data/features'))
    parser.add_argument('--labels', type=Path, default=Path('data/labels'))
    args = parser.parse_args()
    if args.trials < 6 or args.threads < 1:
        parser.error('Use at least six trials and one thread.')
    first = next((args.features / 'train').glob('*/candidates.npz'), None)
    if first is None:
        parser.error('Generate candidate features and aligned labels first.')
    with np.load(first, allow_pickle=False) as data:
        names = data['feature_names'].tolist()
    train = load_trips(args.features, args.labels, 'train', names)
    folds = grouped_folds(train)
    validation_sets = [combine(fold) for fold in folds]

    def objective(trial):
        params = parameters(trial)
        values = []
        for i, validation in enumerate(validation_sets):
            x, y, _, _ = combine([t for j, fold in enumerate(folds) if j != i for t in fold])
            model = estimator(params).fit(x, y)
            values.append(evaluate('boosting', model, validation)[args.objective])
        trial.set_user_attr('fold_scores', values)
        return float(np.mean(values))

    study = optuna.create_study(direction='maximize',
                                sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=5))
    study.enqueue_trial(BASELINE)
    with threadpool_limits(limits=args.threads):
        study.optimize(objective, n_trials=args.trials, n_jobs=1)
        student = load_trips(args.features, args.labels, 'student_test', names)
        if {t[0] for t in train} & {t[0] for t in student}:
            raise ValueError('Train and student-test trip IDs overlap.')
        test = combine(student)
        x, y, _, _ = combine(train)
        baseline = estimator(BASELINE).fit(x, y)
        tuned = estimator(study.best_params).fit(x, y)
        held_out = {
            'baseline': evaluate('boosting', baseline, test),
            'tuned': evaluate('boosting', tuned, test),
        }
    report = dict(model='boosting', objective=args.objective, seed=SEED,
                  baseline_params=BASELINE, best_params=study.best_params,
                  best_trial=study.best_trial.number, best_cv=float(study.best_value),
                  fold_trip_ids=[[trip[0] for trip in fold] for fold in folds],
                  optuna_version=optuna.__version__, sklearn_version=sklearn.__version__,
                  feature_names=names, student_test=held_out,
                  trials=[dict(number=t.number, params=t.params, value=t.value,
                               state=t.state.name, fold_scores=t.user_attrs.get('fold_scores'))
                          for t in study.trials])
    report_path = Path('results/optimization/boosting_bayesian.json')
    model_path = Path('models/boosting_bayesian.pkl')
    report_path.parent.mkdir(parents=True, exist_ok=True)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    with model_path.open('wb') as handle:
        pickle.dump(dict(model=tuned, feature_names=names,
                         sklearn_version=sklearn.__version__), handle)
    print(f'Results: {report_path} | Model: {model_path}')

In [ ]:
if RUN_HGB_TUNING:
    run_stage(hgb_tuning_main, '--trials', HGB_TRIALS)

## 6. Compare saved model results

Top-1 measures whether the highest-ranked candidate matches the aligned road label for each scored GPS observation.

In [ ]:
print(f"{'Model':<18} {'CV top-1':>10} {'Test top-1':>11} {'Test macro F1':>14}")
for path in sorted((ROOT / 'results/training').glob('*.json')):
    report = json.loads(path.read_text(encoding='utf-8'))
    test = report.get('student_test', report)
    if 'top1_accuracy' in test:
        cv = report.get('cv_top1')
        cv_text = f'{cv:.2%}' if cv is not None else '—'
        print(f"{report.get('model', path.stem):<18} {cv_text:>10} "
              f"{test['top1_accuracy']:>11.2%} {test['macro_f1']:>14.2%}")

tuning_report = ROOT / 'results/optimization/boosting_bayesian.json'
if tuning_report.is_file():
    tuned = json.loads(tuning_report.read_text(encoding='utf-8'))
    print('HGB best parameters:', tuned['best_params'])
    print('HGB tuned student-test top-1:', f"{tuned['student_test']['tuned']['top1_accuracy']:.2%}")

## 7. Decode and visualize a student-test route

This stage uses the logistic model as the demo scorer. Change `TRIP_ID` to inspect another processed student-test trip. The plotted lines show the selected road segments.

In [ ]:
"""Decode directed road candidates with distance-based Viterbi transitions."""
import heapq

import numpy as np


def bounded_routes(graph, source, targets, limit):
    queue = [(0.0, int(source))]
    distances = {int(source): 0.0}
    found = {}
    while queue and len(found) < len(targets):
        distance, node = heapq.heappop(queue)
        if distance != distances[node]:
            continue
        if node in targets:
            found[node] = distance
        for end, length in graph.get(node, ()):
            total = distance + length
            if total <= limit and total < distances.get(end, np.inf):
                distances[end] = total
                heapq.heappush(queue, (total, end))
    return found


def decode(inputs, candidates, probabilities, max_gap_s=15.0):
    """Return one candidate row per observation and boundaries between route pieces."""
    offsets = candidates['offsets']
    rows = candidates['candidate_arc_ids']
    arcs = inputs['arcs']
    lengths = inputs['arc_length_m']
    fraction = candidates['features'][:, 2]
    graph = {}
    for (start, end), length in zip(arcs, lengths):
        graph.setdefault(int(start), []).append((int(end), float(length)))
    logp = np.log(np.clip(probabilities, 1e-8, 1.0))
    back = np.full(len(rows), -1, dtype=np.int64)
    path = np.empty(len(offsets) - 1, dtype=np.int64)
    boundaries = [0]
    previous = logp[:offsets[1]]

    def close_piece(end, scores):
        best = offsets[end] + int(scores.argmax())
        for i in range(end, boundaries[-1] - 1, -1):
            path[i] = best
            best = back[best]

    for i in range(1, len(path)):
        left, middle, right = offsets[i - 1:i + 2]
        dt = inputs['timestamps'][i] - inputs['timestamps'][i - 1]
        step = np.linalg.norm(inputs['gps_xy'][i] - inputs['gps_xy'][i - 1])
        current = np.full(right - middle, -np.inf)
        if 0 < dt <= max_gap_s:
            limit = float(min(1200, max(80, step + 100, dt * 55)))
            tolerance = max(20, 0.4 * step)
            targets = set(arcs[rows[middle:right], 0].tolist())
            cache = {}
            for j in range(left, middle):
                if not np.isfinite(previous[j - left]):
                    continue
                old = rows[j]
                end = int(arcs[old, 1])
                if end not in cache:
                    cache[end] = bounded_routes(graph, end, targets, limit)
                for k in range(middle, right):
                    new = rows[k]
                    road = ((1 - fraction[j]) * lengths[old]
                            + cache[end].get(int(arcs[new, 0]), np.inf)
                            + fraction[k] * lengths[new])
                    if old == new and fraction[k] >= fraction[j]:
                        road = min(road, (fraction[k] - fraction[j]) * lengths[old])
                    if road <= limit:
                        score = previous[j - left] - abs(road - step) / tolerance
                        if score > current[k - middle]:
                            current[k - middle] = score
                            back[k] = j
        if np.isfinite(current).any():
            previous = current + logp[middle:right]
            previous -= np.max(previous)
        else:
            close_piece(i - 1, previous)
            boundaries.append(i)
            previous = logp[middle:right]
    close_piece(len(path) - 1, previous)
    return path, boundaries

In [ ]:
import re
from functools import lru_cache

class Demo:
    def __init__(self, data, labels, model, split):
        self.data, self.labels = data / split, labels / split
        self.trip_ids = sorted(p.parent.name for p in self.data.glob('*/inputs.npz')
                               if re.fullmatch(r'[0-9]{8}', p.parent.name))
        if not self.trip_ids:
            raise ValueError(f'No preprocessed trips in {self.data}')
        self.split = split
        with np.load(model, allow_pickle=False) as file:
            self.model = dict(file)

    @lru_cache(maxsize=4)
    def match(self, trip):
        if trip not in self.trip_ids:
            raise KeyError(trip)
        with np.load(self.data / trip / 'inputs.npz', allow_pickle=False) as file:
            inputs = dict(file)
        candidates = generate(inputs)
        if not np.array_equal(candidates['feature_names'], self.model['feature_names']):
            raise ValueError('Model feature schema differs from candidate features')
        probability = scores('logistic', self.model, candidates['features'])
        path, breaks = decode(inputs, candidates, probability)
        offsets = candidates['offsets']
        picks = [left + int(probability[left:right].argmax())
                 for left, right in zip(offsets[:-1], offsets[1:])]
        ids = candidates['candidate_arc_ids']
        selected = ids[path]
        origin = inputs['projection_origin_lonlat']
        metres_per_degree = (6371008.8 * np.pi / 180) * np.array([np.cos(np.radians(origin[1])), 1])
        snapped = candidates['snapped_xy'][path] / metres_per_degree + origin
        selected_geometry = inputs['nodes_lonlat'][inputs['arcs'][selected]]
        review_arcs = np.unique(ids)
        roads = inputs['nodes_lonlat'][inputs['arcs'][review_arcs]]
        result = dict(trip=trip, model='Logistic regression + Viterbi',
                      observations=len(selected), pieces=len(breaks), breaks=breaks,
                      gps=inputs['gps_lonlat'].tolist(), snapped=snapped.tolist(),
                      roads=roads.tolist(), matched_segments=selected_geometry.tolist(),
                      classifier_segments=inputs['nodes_lonlat'][inputs['arcs'][ids[picks]]].tolist(),
                      arc_ids=selected.tolist(), scores=None)
        label_file = self.labels / f'{trip}.npz'
        if self.split == 'student_test' and label_file.is_file():
            with np.load(label_file, allow_pickle=False) as labels:
                if np.array_equal(labels['observation_ids'], inputs['observation_ids']):
                    use = labels['usable']
                    truth = labels['arc_ids'][use]
                    if len(truth):
                        result['scores'] = dict(scored=len(truth),
                            classifier=float(np.mean(ids[picks][use] == truth)),
                            decoded=float(np.mean(selected[use] == truth)))
        return result

In [ ]:
"""Compare the logistic classifier and decoder on the student test trips."""
import json



def decoder_evaluation_main():
    demo = Demo(ROOT / 'data/processed', ROOT / 'data/labels',
                ROOT / 'models/logistic.npz', 'student_test')
    rows = []
    for trip in demo.trip_ids:
        result = demo.match(trip)
        if result['scores'] is None:
            raise ValueError(f'Missing aligned scores for {trip}')
        rows.append(dict(trip=trip, observations=result['observations'],
                         pieces=result['pieces'], **result['scores']))
        print(trip, result['scores'], flush=True)
    total = sum(row['scored'] for row in rows)
    report = {
        'model': 'logistic',
        'split': 'student_test',
        'scored_observations': total,
        'classifier_top1': sum(round(r['classifier'] * r['scored']) for r in rows) / total,
        'decoded_top1': sum(round(r['decoded'] * r['scored']) for r in rows) / total,
        'trips': rows,
    }
    output = ROOT / 'results/metrics/logistic_decoder.json'
    output.parent.mkdir(parents=True, exist_ok=True)
    output.write_text(json.dumps(report, indent=2) + '\n', encoding='utf-8')
    print(json.dumps({key: value for key, value in report.items() if key != 'trips'}))

In [ ]:
TRIP_ID = None  # None uses the first student-test trip.
if RUN_ROUTE_PREVIEW or RUN_DECODER_EVALUATION:
    if not (ROOT / 'models/logistic.npz').is_file():
        raise FileNotFoundError('Run the logistic regression cell before decoding.')
if RUN_DECODER_EVALUATION:
    decoder_evaluation_main()
if RUN_ROUTE_PREVIEW:
    import matplotlib.pyplot as plt
    from matplotlib.collections import LineCollection

    demo = Demo(ROOT / 'data/processed', ROOT / 'data/labels',
                ROOT / 'models/logistic.npz', 'student_test')
    route = demo.match(TRIP_ID or demo.trip_ids[0])
    gps = np.asarray(route['gps'])
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.add_collection(LineCollection(route['roads'], colors='#a3b2bc', linewidths=1,
                                     label='Candidate roads'))
    ax.add_collection(LineCollection(route['matched_segments'], colors='#087f6c', linewidths=2,
                                     label='Decoded roads'))
    ax.scatter(gps[:, 0], gps[:, 1], s=5, alpha=0.45, color='#315bdc', label='GPS')
    ax.set_aspect(1 / np.cos(np.radians(gps[:, 1].mean())))
    ax.autoscale()
    ax.set(xlabel='Longitude', ylabel='Latitude', title=f"Student-test trip {route['trip']}")
    ax.legend()
    plt.show()
    print('Route pieces:', route['pieces'], '| Scores:', route['scores'])

Outputs are saved under `/content/pathsnap/results` and `/content/pathsnap/models`. Download them from the Colab Files sidebar before ending the session. KNN artifacts retain fitted training examples, so share them only with the intended team.